# Lagos building footprints with Google Earth Engine

This notebook clips Google Research Open Buildings V3 polygons to one Lagos LGA and exports the result. It uses an existing model-derived footprint layer; it does not train a new model.

The default study area is Ikeja LGA. Change `AREA_NAME` to another exact name printed below. The GeoJSON and a one-row summary CSV are saved in `My Drive/Lagos_Open_Buildings/`.

Open Buildings V3 is a model-derived 2023 footprint dataset, not an authoritative cadastral inventory or fresh validation. The default confidence cutoff is 0.75. No outputs are included until you run this notebook with your Earth Engine account.

In [ ]:
%pip -q install earthengine-api folium
import ee
import folium
import time
from IPython.display import display

EE_PROJECT_ID = input('Earth Engine Google Cloud Project ID: ').strip()
if not EE_PROJECT_ID:
    raise ValueError('Enter a Google Cloud Project ID. Do not enter a password, token, or key.')
ee.Authenticate()  # Sign in with your Google account when Colab prompts you.
ee.Initialize(project=EE_PROJECT_ID)
print('Earth Engine initialized.')

## Choose a Lagos boundary

The default uses FAO GAUL 2025 second-level administrative boundaries. Review the printed names and select one LGA to keep the query and export manageable.

In [ ]:
admin2 = ee.FeatureCollection('FAO/GAUL/2025/level2')
lagos_areas = (admin2
    .filter(ee.Filter.eq('GAUL0_NAME', 'Nigeria'))
    .filter(ee.Filter.stringContains('GAUL1_NAME', 'Lagos')))
area_names = sorted(set(lagos_areas.aggregate_array('GAUL2_NAME').getInfo()))
print('Lagos second-level areas:', area_names)
AREA_NAME = input('Area name (press Enter for Ikeja): ').strip() or 'Ikeja'
selected = lagos_areas.filter(ee.Filter.eq('GAUL2_NAME', AREA_NAME))
if selected.size().getInfo() != 1:
    raise ValueError(f'{AREA_NAME!r} did not match one boundary. Choose an exact name from the printed list.')
boundary = ee.Feature(selected.first())
aoi = boundary.geometry()
AREA_KM2 = aoi.area(1).divide(1e6).getInfo()
print('Selected:', AREA_NAME, '| area km²:', round(AREA_KM2, 2))

In [ ]:
# Count and preview the filtered footprint layer before exporting.
MIN_CONFIDENCE = 0.75
source_buildings = (ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons')
    .filterBounds(aoi)
    .filter(ee.Filter.gte('confidence', MIN_CONFIDENCE)))
building_count = source_buildings.size().getInfo()
print(f'{building_count:,} footprints at confidence >= {MIN_CONFIDENCE}')

# Paint server-side map tiles; this previews the features without downloading them to Colab.
painted = ee.Image().byte().paint(source_buildings, 1).selfMask().clip(aoi).visualize(palette=['00A86B'])
tile_url = painted.getMapId()['tile_fetcher'].url_format
center = aoi.centroid(1).coordinates().getInfo()
preview = folium.Map(location=[center[1], center[0]], zoom_start=12, control_scale=True)
folium.TileLayer(tiles=tile_url, attr='Google Earth Engine / Google Open Buildings', name='Open Buildings V3', overlay=True).add_to(preview)
folium.LayerControl().add_to(preview)
display(preview)

In [ ]:
# Export both the clipped footprints and a compact summary for reporting.
from datetime import datetime, timezone
def clip_building(feature):
    feature = ee.Feature(feature)
    clipped_geometry = feature.geometry().intersection(aoi, ee.ErrorMargin(1))
    return ee.Feature(clipped_geometry).set({
        'confidence': feature.get('confidence'),
        'clipped_area_m2': clipped_geometry.area(1)
    })

export_features = source_buildings.map(clip_building)
description = f'{AREA_NAME}_open_buildings_conf075'
geojson_task = ee.batch.Export.table.toDrive(
    collection=export_features,
    description=description,
    folder='Lagos_Open_Buildings',
    fileNamePrefix=description,
    fileFormat='GeoJSON')
summary = ee.FeatureCollection([ee.Feature(None, {
    'boundary_name': AREA_NAME,
    'boundary_area_km2': AREA_KM2,
    'boundary_source': 'FAO GAUL 2025 level 2',
    'footprint_dataset': 'Google Research Open Buildings V3',
    'source_inference_year': 2023,
    'minimum_confidence': MIN_CONFIDENCE,
    'filtered_footprint_count': building_count,
    'export_started_utc': datetime.now(timezone.utc).isoformat(),
    'source_license': 'CC BY 4.0'
})])
summary_task = ee.batch.Export.table.toDrive(
    collection=summary,
    description=f'{description}_summary',
    folder='Lagos_Open_Buildings',
    fileNamePrefix=f'{description}_summary',
    fileFormat='CSV')
geojson_task.start()
summary_task.start()
print('Export tasks started. Waiting for Earth Engine...')
while geojson_task.active() or summary_task.active():
    print('GeoJSON:', geojson_task.status().get('state'), '| summary:', summary_task.status().get('state'))
    time.sleep(20)
geojson_status, summary_status = geojson_task.status(), summary_task.status()
print('GeoJSON status:', geojson_status)
print('Summary status:', summary_status)
if geojson_status.get('state') != 'COMPLETED' or summary_status.get('state') != 'COMPLETED':
    raise RuntimeError(f'One or more Earth Engine exports failed: {geojson_status} | {summary_status}')
print(f'Finished. Find {description}.geojson and {description}_summary.csv in My Drive/Lagos_Open_Buildings.')

## Outputs and attribution

The GeoJSON contains clipped footprint geometries, source confidence and clipped area in square metres. The summary CSV contains the selected LGA, boundary area, confidence threshold, filtered footprint count, source year, and export timestamp. Earth Engine writes both files to Google Drive.

Use the actual summary CSV values and the map preview in your presentation. Use measured values from the summary CSV; do not insert placeholders. Attribute Google Research Open Buildings V3 (CC BY 4.0) and FAO GAUL 2025 (CC BY 4.0) with any reused output. These polygons are model-derived and should not be described as a surveyed or complete building register.